# Data Preprocessing
Colab's local disk only has about 30-60GB of free space, which isn't enough to unzip the full AI Hub dataset. So resizing is instead performed using a local machine's disk.

## 1. Analyzing the Downloaded Data
The data downloaded from AI Hub has the following structure:


```bash
Landmark Images (랜드마크 이미지)
├─Training
│  └─Sejong Special Self-Governing City (세종특별자치시)
│          [Label] Sejong.zip  ([라벨]세종특별자치시.zip)
│          [Source] Sejong_001.zip  ([원천]세종특별자치시_001.zip)
└─Validation
    └─Sejong Special Self-Governing City (세종특별자치시)
            [Label] Sejong.zip  ([라벨]세종특별자치시.zip)
            [Source] Sejong_001.zip  ([원천]세종특별자치시_001.zip)
```
The test and validation data are already separated, so they are used as-is. The `[라벨]` ("Label") files are JSON files containing metadata, which are not needed at this stage and are ignored. In other words, only the `[원천]` ("Source") files need to be unzipped and resized.

> Note: the actual folder and file names in the downloaded dataset are in Korean (e.g. `랜드마크 이미지`, `세종특별자치시`, `[원천]...`). The code below matches those Korean names directly since it operates on the real dataset -- they are intentionally left untranslated so the code keeps working against the original files.

## 2. Unzipping the Data
The regional name in the intermediate folder is not needed, so it is ignored; the directories are restructured and the archives are extracted.

In [3]:
import os, glob

# NOTE: this path matches the actual (Korean-named) folder from the AI Hub dataset -- kept as-is so the code works against the real files
base_dir = '/content/랜드마크 이미지/'
extract_dir = '/content/extracts'

In [ ]:

for dtype in os.listdir(base_dir):
    dst_dir = os.path.join(extract_dir, dtype)
    os.makedirs(dst_dir, exist_ok=True)
    for file_name in glob.glob(os.path.join(base_dir, dtype) + '/**/*원천*', recursive=True):
        file_name = '\'' + file_name + '\''
        !unzip -o -qq {file_name} -d {dst_dir}

## 3. RESIZE
The structure of the unzipped data is as below. File extensions are `JPG` or `jpg`, so matching files are found and
resized to 0.1x with OpenCV before being saved.

```bash
6.25gyeokjeonji-gaemigogae (6.25격전지개미고개, a Korean War battle-site landmark)
  └─6.25격전지개미고개_001_40613647.JPG
    6.25격전지개미고개_003_40613648.JPG
    6.25격전지개미고개_004_40613649.JPG
    6.25격전지개미고개_005_40613650.JPG
    6.25격전지개미고개_006_40613651.JPG
    (...)
```
> Note: file names come directly from the source dataset and are in Korean; shown as-is for accuracy, with an English gloss of the folder name above.

In [5]:
import cv2
from tqdm.notebook import tqdm

resized_dir = '/content/resizeds'

In [ ]:
for dtype in os.listdir(extract_dir):
    src_dir = os.path.join(extract_dir, dtype)
    dst_dir = os.path.join(resized_dir, dtype)
    for cls in os.listdir(src_dir):
        os.makedirs(os.path.join(dst_dir, cls), exist_ok=True)
    
    src_list = glob.glob('**/*.JPG', root_dir=os.path.join(extract_dir, dtype))
    print('Dataset Type: {}'.format(dtype))
    for fname in tqdm(src_list):
        img = cv2.imread(os.path.join(src_dir, fname))
        resized_img = cv2.resize(img, (0, 0), fx=0.1, fy=0.1, interpolation=cv2.INTER_AREA)
        cv2.imwrite(os.path.join(dst_dir, fname), resized_img)
    


## 4. Compression
To use a GPU, the data is re-compressed into a zip file so it can be uploaded to Colab.

In [ ]:
!cd '/content'
!zip -r 'resize_0.1.zip' './resizeds'